# Visualising elfe3D_GPR Results - Finals of May

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pyvista as pv
from scipy.interpolate import griddata

base_folder = "F:\\Projects\\EMGeoInversion\\Tests_Thesis\\May-Final"
postprocess_folder = os.path.join(base_folder, "postprocess")
analytical_folder = os.path.join(base_folder, "semi-analytic_100MHz")

z_value = 0.01
z_tol = 0.05
xlim=(-3, 6.5)
ylim=(-3, 6.5)
xlim_base = (-5, 5)
ylim_base = (-5, 5)
xlim_a = (-1.5, 5)
ylim_a = (-1.5, 5)
z_tol=0.05
dpi=300
grid_resolution=3000
amplitude_cmap='Blues_r'
phase_cmap='RdBu_r'

component = 'Ex' 

In [12]:
vtk_file_1 = os.path.join(base_folder, "GPR_model_air_half_PML_surf_feng.1.vtk")
elfe_vtk_1 = pv.read(vtk_file_1)

vtk_file_2 = os.path.join(base_folder, "GPR_model_air_qua_PML_025_feng.1.vtk")
elfe_vtk_2 = pv.read(vtk_file_2)

vtk_file_3 = os.path.join(base_folder, "GPR_model_air_qua_PML_surf_ber.1.vtk")
elfe_vtk_3 = pv.read(vtk_file_3)

vtk_file_4 = os.path.join(base_folder, "GPR_model_air_qua_PML_surf_ber2.1.vtk")
elfe_vtk_4 = pv.read(vtk_file_4)

vtk_file_5 = os.path.join(base_folder, "GPR_model_air_qua_PML_surf_feng.1.vtk")
elfe_vtk_5 = pv.read(vtk_file_5)

elfe_vtk_all = [elfe_vtk_1, elfe_vtk_2, elfe_vtk_3, elfe_vtk_4, elfe_vtk_5]


In [13]:
# Extracting Plot Data
amp_field_elfe = f"Real_{component}"
phase_field_elfe = f"Imag_{component}"

# --- elfe3D Cell Centers and Scalars for all entries in elfe_vtk_all ---
amp_elfe_grid_all = []
phase_elfe_grid_all = []

for elfe_vtk_entry in elfe_vtk_all:
    # --- elfe3D Cell Centers and Scalars ---
    cell_centers = elfe_vtk_entry.cell_centers().points
    real_elfe = elfe_vtk_entry.cell_data[f"Real_{component}"]
    imag_elfe = elfe_vtk_entry.cell_data[f"Imag_{component}"]
    complex_elfe = real_elfe + 1j * imag_elfe

    # --- Filter to z-slice ---
    mask = np.abs(cell_centers[:, 2] - z_value) <= z_tol
    x_elfe = cell_centers[mask, 0]
    y_elfe = cell_centers[mask, 1]
    complex_elfe_slice = complex_elfe[mask]

    xi = np.linspace(xlim[0], xlim[1], grid_resolution)
    yi = np.linspace(ylim[0], ylim[1], grid_resolution)
    grid_x, grid_y = np.meshgrid(xi, yi)

    # --- Interpolate elfe3D fields to regular grid ---
    complex_elfe_grid = griddata((x_elfe, y_elfe), complex_elfe_slice, (grid_x, grid_y), method='linear')
    amp_elfe_grid = np.log10(np.clip(np.abs(complex_elfe_grid), 1e-60, None))
    phase_elfe_grid = np.angle(complex_elfe_grid)  # returns phase in radians (−π to π)

    amp_elfe_grid_all.append(amp_elfe_grid)
    phase_elfe_grid_all.append(phase_elfe_grid)

phase_clim = [-np.pi, np.pi]
amp_min = np.floor(np.nanmin(amp_elfe_grid_all[0]))
amp_max = np.ceil(np.nanmax(amp_elfe_grid_all[0]))
amp_clim = [amp_min, amp_max]

# Analytical
# Mask grid_x and grid_y to be within xlim_a and ylim_a
mask_x = (grid_x >= xlim_a[0]) & (grid_x <= xlim_a[1])
mask_y = (grid_y >= ylim_a[0]) & (grid_y <= ylim_a[1])
mask = mask_x & mask_y

# Mask grid_x and grid_y to be within xlim_a and ylim_a, and flatten to get valid points
valid_mask = mask.flatten()
grid_x_bounded = grid_x.flatten()[valid_mask]
grid_y_bounded = grid_y.flatten()[valid_mask]

analytical_file = os.path.join(analytical_folder, "air_z0.01_100MHz_xmax5.csv")
analytical_slices = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

x = analytical_slices[:, 0]
y = analytical_slices[:, 1]

x_unique = np.unique(x)
y_unique = np.unique(y)
nx, ny = len(x_unique), len(y_unique)

if len(x) != nx * ny:
    raise ValueError("Mismatch in analytical grid. Expected meshgrid-style structure.")

if component == 'Ex':
    real_index = 4
    imag_index = 5
elif component == 'Ey':
    real_index = 8
    imag_index = 9
elif component == 'Ez':
    real_index = 12
    imag_index = 13
else:
    raise ValueError("Invalid component. Choose from 'Ex', 'Ey', or 'Ez'.")

real_analytical = analytical_slices[:, real_index].reshape((ny, nx))
imag_analytical = analytical_slices[:, imag_index].reshape((ny, nx))

# ----------------------------------------------------------------------
# Extra Process Based on Inference
# ----------------------------------------------------------------------
real_analytical = -1 * real_analytical
imag_analytical = -1 * imag_analytical

complex_analytical = real_analytical + 1j * imag_analytical

# Interpolation
complex_analytical_grid = griddata((x, y), complex_analytical.flatten(), (grid_x_bounded, grid_y_bounded), method='linear')
amp_analytical_grid = np.log10(np.clip(np.abs(complex_analytical_grid), 1e-60, None))
phase_analytical_grid = np.angle(complex_analytical_grid)  # returns phase in radians (−π to π)
real_analytical_grid = griddata((x, y), real_analytical.flatten(), (grid_x_bounded, grid_y_bounded), method='linear')
imag_analytical_grid = griddata((x, y), imag_analytical.flatten(), (grid_x_bounded, grid_y_bounded), method='linear')

# Resize analytical grids to match grid_x and grid_y shape, filling extra entries with NaN
amp_analytical_grid_full = np.full(grid_x.shape, np.nan)
phase_analytical_grid_full = np.full(grid_x.shape, np.nan)
real_analytical_grid_full = np.full(grid_x.shape, np.nan)
imag_analytical_grid_full = np.full(grid_x.shape, np.nan)

amp_analytical_grid_full.flat[valid_mask] = amp_analytical_grid
phase_analytical_grid_full.flat[valid_mask] = phase_analytical_grid
real_analytical_grid_full.flat[valid_mask] = real_analytical_grid
imag_analytical_grid_full.flat[valid_mask] = imag_analytical_grid

amp_analytical_grid = amp_analytical_grid_full
phase_analytical_grid = phase_analytical_grid_full
real_analytical_grid = real_analytical_grid_full
imag_analytical_grid = imag_analytical_grid_full

# --- Plotting ---
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
fig, axs = plt.subplots(2, n_entries + 1, figsize=(6.5 * (n_entries + 1), 12), dpi=dpi)

amp_text = r'$\log_{10}(|' + component + r'|)$: '
amp_text_list = [r'$\lambda/2$ PML, Ground, Feng', 
                 r'$\lambda/4$ PML, 2.5 cm Above, Feng', 
                 r'$\lambda/4$ PML, Ground, Berenger1', 
                 r'$\lambda/4$ PML, Ground, Berenger2', 
                 r'$\lambda/4$ PML, Ground, Feng']
phase_text = 'Phase: '
phase_text_list = amp_text_list.copy()

for i in range(n_entries):
    amp_text_list[i] = amp_text + amp_text_list[i]
    phase_text_list[i] = phase_text + phase_text_list[i]

for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=(xlim[0], xlim[1], ylim[0], ylim[1]), origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
    axs[0, i].set_title(amp_text_list[i], fontsize=20, fontweight='bold')
    axs[0, i].set_xlabel("x", fontsize=18)
    axs[0, i].set_ylabel("y", fontsize=18)
    axs[0, i].tick_params(axis='both', which='major', labelsize=18)
    rect1 = plt.Rectangle((-1.5, -1.5), 6.5, 6.5, linewidth=4, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=(xlim[0], xlim[1], ylim[0], ylim[1]), origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(phase_text_list[i], fontsize=20, fontweight='bold')
    axs[1, i].set_xlabel("x", fontsize=18)
    axs[1, i].set_ylabel("y", fontsize=18)
    axs[1, i].tick_params(axis='both', which='major', labelsize=18)
    rect2 = plt.Rectangle((-1.5, -1.5), 6.5, 6.5, linewidth=4, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=18)

# Analytical solution (rightmost column)
im2 = axs[0, n_entries].imshow(amp_analytical_grid, extent=(xlim[0], xlim[1], ylim[0], ylim[1]), origin='lower',
    cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
axs[0, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[0, n_entries].set_xlabel("x", fontsize=18)
axs[0, n_entries].set_ylabel("y", fontsize=18)
axs[0, n_entries].tick_params(axis='both', which='major', labelsize=18)
rect1 = plt.Rectangle((-1.5, -1.5), 6.5, 6.5, linewidth=4, edgecolor='red', facecolor='none', linestyle='--')
axs[0, n_entries].add_patch(rect1)
cbar2 = fig.colorbar(im2, ax=axs[0, n_entries], fraction=0.046, pad=0.02)
cbar2.ax.tick_params(labelsize=18)

im3 = axs[1, n_entries].imshow(phase_analytical_grid, extent=(xlim[0], xlim[1], ylim[0], ylim[1]), origin='lower',
    cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
axs[1, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[1, n_entries].set_xlabel("x", fontsize=18)
axs[1, n_entries].set_ylabel("y", fontsize=18)
axs[1, n_entries].tick_params(axis='both', which='major', labelsize=18)
rect2 = plt.Rectangle((-1.5, -1.5), 6.5, 6.5, linewidth=4, edgecolor='black', facecolor='none', linestyle='--')
axs[1, n_entries].add_patch(rect2)
cbar3 = fig.colorbar(im3, ax=axs[1, n_entries], fraction=0.046, pad=0.02)
cbar3.ax.tick_params(labelsize=18)

title = f'Comparison of Various PML Implementation Methods With the Simple Exact Decay Function, {component} Component of an Electric Dipole Field in Air, 2D Slice Taken z = {z_value:.3f} m Below the Source'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
plt.tight_layout(rect=[0, 0, 1, 0.89])  # leave even more space for the title
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# Save figure
filename_combined = f"1_theory_numerical_comparison.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)

if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)

fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)

In [14]:
# --- Clipped Plotting ---
clip=4
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
fig, axs = plt.subplots(2, n_entries + 1, figsize=(6.5 * (n_entries + 1), 12), dpi=dpi)

amp_text = r'$\log_{10}(|' + component + r'|)$: '
amp_text_list = [r'$\lambda/2$ PML, Ground, Feng', 
                 r'$\lambda/4$ PML, 2.5 cm Above, Feng', 
                 r'$\lambda/4$ PML, Ground, Berenger1', 
                 r'$\lambda/4$ PML, Ground, Berenger2', 
                 r'$\lambda/4$ PML, Ground, Feng']

phase_text = 'Phase: '
phase_text_list = amp_text_list.copy()

for i in range(n_entries):
    amp_text_list[i] = amp_text + amp_text_list[i]
    phase_text_list[i] = phase_text + phase_text_list[i]

for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=(xlim[0], xlim[1], ylim[0], ylim[1]), origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max-clip)
    axs[0, i].set_title(amp_text_list[i], fontsize=20, fontweight='bold')
    axs[0, i].set_xlabel("x", fontsize=18)
    axs[0, i].set_ylabel("y", fontsize=18)
    axs[0, i].tick_params(axis='both', which='major', labelsize=18)
    rect1 = plt.Rectangle((-1.5, -1.5), 6.5, 6.5, linewidth=4, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=(xlim[0], xlim[1], ylim[0], ylim[1]), origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(phase_text_list[i], fontsize=20, fontweight='bold')
    axs[1, i].set_xlabel("x", fontsize=18)
    axs[1, i].set_ylabel("y", fontsize=18)
    axs[1, i].tick_params(axis='both', which='major', labelsize=18)
    rect2 = plt.Rectangle((-1.5, -1.5), 6.5, 6.5, linewidth=4, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=18)

# Analytical solution (rightmost column)
im2 = axs[0, n_entries].imshow(amp_analytical_grid, extent=(xlim[0], xlim[1], ylim[0], ylim[1]), origin='lower',
    cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max-clip)
axs[0, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[0, n_entries].set_xlabel("x", fontsize=18)
axs[0, n_entries].set_ylabel("y", fontsize=18)
axs[0, n_entries].tick_params(axis='both', which='major', labelsize=18)
rect1 = plt.Rectangle((-1.5, -1.5), 6.5, 6.5, linewidth=4, edgecolor='red', facecolor='none', linestyle='--')
axs[0, n_entries].add_patch(rect1)
cbar2 = fig.colorbar(im2, ax=axs[0, n_entries], fraction=0.046, pad=0.02)
cbar2.ax.tick_params(labelsize=18)

im3 = axs[1, n_entries].imshow(phase_analytical_grid, extent=(xlim[0], xlim[1], ylim[0], ylim[1]), origin='lower',
    cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
axs[1, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[1, n_entries].set_xlabel("x", fontsize=18)
axs[1, n_entries].set_ylabel("y", fontsize=18)
axs[1, n_entries].tick_params(axis='both', which='major', labelsize=18)
rect2 = plt.Rectangle((-1.5, -1.5), 6.5, 6.5, linewidth=4, edgecolor='black', facecolor='none', linestyle='--')
axs[1, n_entries].add_patch(rect2)
cbar3 = fig.colorbar(im3, ax=axs[1, n_entries], fraction=0.046, pad=0.02)
cbar3.ax.tick_params(labelsize=18)

title = f'Clipped Comparison of Various PML Implementation Methods With the Simple Exact Decay Function, {component} Component of an Electric Dipole Field in Air, 2D Slice Taken z = {z_value:.3f} m Below the Source'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
plt.tight_layout(rect=[0, 0, 1, 0.89])  # leave even more space for the title
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# Save figure
filename_combined = f"2_theory_numerical_comparison_clipped.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)

if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)

fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)

In [15]:
n_entries = len(elfe_vtk_all)
extents = (-1.5, 5, -1.5, 5)  # Extents for the images
fig, axs = plt.subplots(4, n_entries, figsize=(6.5 * n_entries, 24), dpi=dpi)

amp_text_list = [
    r'$\lambda/2$ PML, Ground, Feng',
    r'$\lambda/4$ PML, 2.5 cm Above, Feng',
    r'$\lambda/4$ PML, Ground, Berenger1',
    r'$\lambda/4$ PML, Ground, Berenger2',
    r'$\lambda/4$ PML, Ground, Feng'
]
for i, label in enumerate(amp_text_list):
    axs[0, i].set_title(f'Log of Ratio of Amplitudes: {label}', fontsize=18, fontweight='bold')
    axs[1, i].set_title(f'Normalised Diff, Amplitudes: {label}', fontsize=18, fontweight='bold')
    axs[2, i].set_title(f'Diff Phase: {label}', fontsize=18, fontweight='bold')
    axs[3, i].set_title(f'% Diff Phase: {label}', fontsize=18, fontweight='bold')

for i in range(n_entries):
    diff_amp = amp_elfe_grid_all[i] - amp_analytical_grid
    diff_amp_percent = ((np.power(10, amp_elfe_grid_all[i]) - np.power(10, amp_analytical_grid)) / np.power(10, amp_analytical_grid)) * 100
    diff_phase = phase_elfe_grid_all[i] - phase_analytical_grid
    diff_phase_percent = (diff_phase / phase_analytical_grid) * 100

    diff_amp_lim = np.nanmax(np.abs(diff_amp))
    diff_phase_lim = np.nanmax(np.abs(diff_phase))
    diff_amp_percent_lim = 100
    diff_phase_percent_lim = 100

    extent = (xi.min(), xi.max(), yi.min(), yi.max())
    cmap = "RdBu"

    im0 = axs[0, i].imshow(diff_amp, extent=extents, origin='lower', cmap=cmap, vmin=-diff_amp_lim, vmax=diff_amp_lim)
    fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    im1 = axs[1, i].imshow(diff_amp_percent, extent=extents, origin='lower', cmap=cmap, vmin=-diff_amp_percent_lim, vmax=diff_amp_percent_lim)
    fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    im2 = axs[2, i].imshow(diff_phase, extent=extents, origin='lower', cmap=cmap, vmin=-diff_phase_lim, vmax=diff_phase_lim)
    fig.colorbar(im2, ax=axs[2, i], fraction=0.046, pad=0.02)
    im3 = axs[3, i].imshow(diff_phase_percent, extent=extents, origin='lower', cmap=cmap, vmin=-diff_phase_percent_lim, vmax=diff_phase_percent_lim)
    fig.colorbar(im3, ax=axs[3, i], fraction=0.046, pad=0.02)

    for row in range(4):
        axs[row, i].set_xlabel("x", fontsize=16)
        axs[row, i].set_ylabel("y", fontsize=16)
        axs[row, i].tick_params(axis='both', which='major', labelsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.suptitle('Difference Between elfe3D and Analytical Solution for Various Air Spaces Above and Below the Source\n2D Slice Taken z = {:.3f} m Below the Source'.format(z_value), fontsize=27, fontweight='bold')

filename_combined = "3_theory_numerical_comparison_diff.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)
if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)
fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)


In [18]:
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
diff_amp_percent_all = []
diff_phase_all = []
std_amp_all = []
std_phase_all = []
amp_text_list = [
    r'$\lambda/2$ PML, Ground, Feng',
    r'$\lambda/4$ PML, 2.5 cm Above, Feng',
    r'$\lambda/4$ PML, Ground, Berenger1',
    r'$\lambda/4$ PML, Ground, Berenger2',
    r'$\lambda/4$ PML, Ground, Feng'
]
phase_text_list = amp_text_list.copy()
for i in range(n_entries):
    diff_amp_percent = ((np.power(10, amp_elfe_grid_all[i]) - np.power(10, amp_analytical_grid)) / np.power(10, amp_analytical_grid)) * 100
    diff_amp_percent = np.clip(diff_amp_percent, -100, 100)
    diff_phase_percent = (phase_elfe_grid_all[i] - phase_analytical_grid) * 100 
    diff_phase_percent = np.clip(diff_phase_percent, -100, 100)
    std_amp_all.append(np.nanstd(diff_amp_percent))
    std_phase_all.append(np.nanstd(diff_phase_percent))

    diff_amp_percent_all.append(diff_amp_percent)
    diff_phase_all.append(diff_phase_percent)

# Plotting the histograms
fig, axs = plt.subplots(2, n_entries, figsize=(6.5 * n_entries, 14), dpi=dpi)
for i in range(n_entries):
    axs[0, i].hist(diff_amp_percent_all[i].flatten(), bins=100, color='blue', alpha=0.7)
    axs[0, i].set_title(f'Amp Diff %: {amp_text_list[i]}', fontsize=18, fontweight='bold')
    axs[0, i].set_xlabel('Difference (%)', fontsize=16)
    axs[0, i].set_ylabel('Frequency', fontsize=16)
    axs[0, i].tick_params(axis='both', which='major', labelsize=16)
    axs[0, i].axvline(std_amp_all[i], color='blue', linestyle='dashed', linewidth=1)
    axs[0, i].axvline(-std_amp_all[i], color='blue', linestyle='dashed', linewidth=1)
    axs[0, i].text(std_amp_all[i], axs[0, i].get_ylim()[1]*0.9, f'STD: {std_amp_all[i]:.2f}', color='blue', fontsize=16)
    axs[0, i].set_xlabel("x", fontsize=16)
    axs[0, i].set_ylabel("y", fontsize=16)
    axs[0, i].tick_params(axis='both', which='major', labelsize=14)

    axs[1, i].hist(diff_phase_all[i].flatten(), bins=100, color='red', alpha=0.7)
    axs[1, i].set_title(f'Phase Diff %: {phase_text_list[i]}', fontsize=18, fontweight='bold')
    axs[1, i].set_xlabel('Difference (%)', fontsize=16)
    axs[1, i].set_ylabel('Frequency', fontsize=16)
    axs[1, i].tick_params(axis='both', which='major', labelsize=16)
    axs[1, i].axvline(std_phase_all[i], color='red', linestyle='dashed', linewidth=1)
    axs[1, i].axvline(-std_phase_all[i], color='red', linestyle='dashed', linewidth=1)
    axs[1, i].text(std_phase_all[i], axs[1, i].get_ylim()[1]*0.9, f'STD: {std_phase_all[i]:.2f}', color='red', fontsize=16)
    axs[1, i].set_xlabel("x", fontsize=16)
    axs[1, i].set_ylabel("y", fontsize=16)
    axs[1, i].tick_params(axis='both', which='major', labelsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.89])
plt.suptitle('Histogram of Differences Between elfe3D and Analytical Solution for Various PML Implementation\nFrom a 2D Slice Taken z = {:.3f} m Below the Source'.format(z_value), fontsize=27, fontweight='bold')
plt.savefig(os.path.join(postprocess_folder, "4_theory_numerical_comparison_histogram.png"), dpi=dpi)
plt.close(fig)


In [2]:
electric_file = os.path.join(base_folder, "out_4_half_PML_surf_feng", "electric_fields_receiver_line.txt")
e_data_elfe_1 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_qua_PML_025_feng", "electric_fields_receiver_line.txt")
e_data_elfe_2 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_qua_PML_surf_ber", "electric_fields_receiver_line.txt")
e_data_elfe_3 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_qua_PML_surf_ber2", "electric_fields_receiver_line.txt")
e_data_elfe_4 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_qua_PML_surf_feng", "electric_fields_receiver_line.txt")
e_data_elfe_5 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_bc_atomic", "electric_fields_receiver_line.txt")
e_data_elfe_6 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_bc_lists", "electric_fields_receiver_line.txt")
e_data_elfe_7 = np.loadtxt(electric_file)

e_data_elfe_all = [e_data_elfe_1, e_data_elfe_2, e_data_elfe_3, e_data_elfe_4, e_data_elfe_5, 
                   e_data_elfe_6, e_data_elfe_7]

labels = [
    r'$\lambda/2$ PML, On Ground, Feng',
    r'$\lambda/4$ PML, 2.5 cm Above, Feng',
    r'$\lambda/4$ PML, On Ground, Berenger1',
    r'$\lambda/4$ PML, On Ground, Berenger2',
    r'$\lambda/4$ PML, On Ground, Feng',
    r'$\lambda/2$ PML, 2m Air, Atomic BC',
    r'$\lambda/2$ PML, 2m Air, Lists BC',
]

num_rec_bs=15
num_rec_ef=15
num_rec_ob=15

# Analytical Data
analytical_file = os.path.join(analytical_folder, "Exx_single_freq_4_100MHz.csv")
analytical_lines = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

r = analytical_lines[:, 0]

real_broadside = analytical_lines[:, 1]
imag_broadside = analytical_lines[:, 2]
abs_broadside = analytical_lines[:, 3]
complex_broadside = real_broadside + 1j * imag_broadside
phase_broadside = np.angle(complex_broadside)  # returns phase in radians (−π to π)

real_45deg = analytical_lines[:, 5]
imag_45deg = analytical_lines[:, 6]
abs_45deg = analytical_lines[:, 7]
complex_45deg = real_45deg + 1j * imag_45deg
phase_45deg = np.angle(complex_45deg)  # returns phase in radians (−π to π)

real_endfire = analytical_lines[:, 9]
imag_endfire = analytical_lines[:, 10]
abs_endfire = analytical_lines[:, 11]
complex_endfire = real_endfire + 1j * imag_endfire
phase_endfire = np.angle(complex_endfire)  # returns phase in radians (−π to π)

def plot_component_receiver_line(component, rec_x, rec_y, abs_data_list, phase_data_list, x_an, y_an, abs_an, phase_an, type_rec, index, purpose, labels):
    _, axes = plt.subplots(1, 2, figsize=(14, 6))
    type_string = 'y' if type_rec == 'Broadside' else 'x' if type_rec == 'Endfire' else 'Radial distance from the dipole in Oblique Configuration, 45°'

    # Style options
    colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728', '#6a0dad', '#8c564b', '#e377c2']
    linestyles = ['--', ':', '--', ':', '--', ':', '--']
    markers = ['o', 's', '^', 'v', 'X', 'D', 'P']
    marker_size = 8     # Bigger markers (default is ~6)
    line_width = 1.5    # Thinner lines (default is ~2)

    for i, (abs_data, phase_data, rec_xi, rec_yi, label) in enumerate(zip(abs_data_list, phase_data_list, rec_x, rec_y, labels)):
        rec_distance = rec_yi if type_rec == 'Broadside' else rec_xi if type_rec == 'Endfire' else np.sqrt(rec_xi**2 + rec_yi**2)
        style_idx = i % len(colors)
        # Absolute value plot
        axes[0].plot(rec_distance, abs_data,
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)
        # Phase plot
        axes[1].plot(rec_distance, phase_data,
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)

    # Analytical solution
    rec_distance_an = np.array(y_an) if type_rec == 'Broadside' else np.array(x_an) if type_rec == 'Endfire' else np.array(x_an)
    axes[0].plot(rec_distance_an, abs_an, linestyle='-', color='black', label='Analytical Solution', linewidth=2)
    axes[1].plot(rec_distance_an, phase_an, linestyle='-', color='black', label='Analytical Solution', linewidth=2)

    axes[0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0].set_ylabel(f'Abs({component})', fontsize=12)
    axes[0].set_title(f'Absolute Value of {component} Field Component', fontsize=18)
    axes[0].set_yscale('log')
    axes[0].grid(True)
    axes[0].legend(loc='upper right')

    axes[1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1].set_ylabel(f'Phase({component}) (degrees)', fontsize=12)
    axes[1].set_title(f'Phase of {component} Field Component', fontsize=18)
    axes[1].grid(True)

    plt.subplots_adjust(top=0.82)
    title = r'Half-Space Model Response - $\epsilon_r = 4$' \
            '\n{} Receiver Line Data of {} Component'.format(type_rec, component)
    plt.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    plot_name = f'{index}_{purpose}_{component}_receiver_line_{type_rec}.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

# Prepare data for each receiver type
rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list = [], [], [], []
rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list = [], [], [], []
rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list = [], [], [], []

for e_data_elfe in e_data_elfe_all:

    rec_x_elfe = e_data_elfe[:, 0]
    rec_y_elfe = e_data_elfe[:, 1]

    real_Ex_elfe = e_data_elfe[:, 4]
    imag_Ex_elfe = e_data_elfe[:, 5]
    abs_Ex_elfe = np.abs(real_Ex_elfe + 1j * imag_Ex_elfe)
    phase_Ex_elfe = np.angle(real_Ex_elfe + 1j * imag_Ex_elfe) # returns phase in radians (−π to π)

    # Broadside
    i_rx_bs = num_rec_ef
    rec_x_bs = rec_x_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_y_bs = rec_y_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    abs_Ex_bs = abs_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    phase_Ex_bs = phase_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_x_bs_list.append(rec_x_bs)
    rec_y_bs_list.append(rec_y_bs)
    abs_Ex_bs_list.append(abs_Ex_bs)
    phase_Ex_bs_list.append(phase_Ex_bs)

    # Endfire
    i_rx_ef = 0
    rec_x_ef = rec_x_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_y_ef = rec_y_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    abs_Ex_ef = abs_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    phase_Ex_ef = phase_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_x_ef_list.append(rec_x_ef)
    rec_y_ef_list.append(rec_y_ef)
    abs_Ex_ef_list.append(abs_Ex_ef)
    phase_Ex_ef_list.append(phase_Ex_ef)

    # Oblique
    i_rx_ob = num_rec_bs + num_rec_ef
    rec_x_ob = rec_x_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_y_ob = rec_y_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    abs_Ex_ob = abs_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    phase_Ex_ob = phase_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_x_ob_list.append(rec_x_ob)
    rec_y_ob_list.append(rec_y_ob)
    abs_Ex_ob_list.append(abs_Ex_ob)
    phase_Ex_ob_list.append(phase_Ex_ob)

# Analytical data
# -------------------------------------
# Got the broadside and endire mixed up, so now we have to fix it by re-assigning to the right
# variables.
# -------------------------------------
x_an_ef = r
y_an_ef = np.zeros_like(r)
abs_Ex_an_ef = abs_broadside
phase_Ex_an_ef = phase_broadside

x_an_bs = np.zeros_like(r)
y_an_bs = r
abs_Ex_an_bs = abs_endfire
phase_Ex_an_bs = phase_endfire

x_an_ob = r
y_an_ob = r
abs_Ex_an_ob = abs_45deg
phase_Ex_an_ob = phase_45deg

# Plot all on the same axes for each receiver type
plot_component_receiver_line('Ex', rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, 
                             x_an_bs, y_an_bs, abs_Ex_an_bs, phase_Ex_an_bs, 'Broadside', 5, '4', labels)
plot_component_receiver_line('Ex', rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, 
                             x_an_ef, y_an_ef, abs_Ex_an_ef, phase_Ex_an_ef, 'Endfire', 6, '4', labels)
plot_component_receiver_line('Ex', rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, 
                             x_an_ob, y_an_ob, abs_Ex_an_ob, phase_Ex_an_ob, 'Oblique, 45°', 7, '4', labels)

In [3]:
electric_file = os.path.join(base_folder, "out_4_9_half_PML_surf_feng", "electric_fields_receiver_line.txt")
e_data_elfe_1 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_9_qua_PML_025_feng", "electric_fields_receiver_line.txt")
e_data_elfe_2 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_9_qua_PML_surf_ber", "electric_fields_receiver_line.txt")
e_data_elfe_3 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_9_qua_PML_surf_ber2", "electric_fields_receiver_line.txt")
e_data_elfe_4 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_9_qua_PML_surf_feng", "electric_fields_receiver_line.txt")
e_data_elfe_5 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_9_bc_atomic", "electric_fields_receiver_line.txt")
e_data_elfe_6 = np.loadtxt(electric_file)

e_data_elfe_all = [e_data_elfe_1, e_data_elfe_2, e_data_elfe_3, e_data_elfe_4, e_data_elfe_5, e_data_elfe_6]

labels = [
    r'$\lambda/2$ PML, On Ground, Feng',
    r'$\lambda/4$ PML, 2.5 cm Above, Feng',
    r'$\lambda/4$ PML, On Ground, Berenger1',
    r'$\lambda/4$ PML, On Ground, Berenger2',
    r'$\lambda/4$ PML, On Ground, Feng',
    r'$\lambda/2$ PML, 2m Air, Atomic BC'
]

num_rec_bs=15
num_rec_ef=15
num_rec_ob=15

# Analytical Data
analytical_file = os.path.join(analytical_folder, "Exx_single_freq_4_9_100MHz.csv")
analytical_lines = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

r = analytical_lines[:, 0]

real_broadside = analytical_lines[:, 1]
imag_broadside = analytical_lines[:, 2]
abs_broadside = analytical_lines[:, 3]
complex_broadside = real_broadside + 1j * imag_broadside
phase_broadside = np.angle(complex_broadside)  # returns phase in radians (−π to π)

real_45deg = analytical_lines[:, 5]
imag_45deg = analytical_lines[:, 6]
abs_45deg = analytical_lines[:, 7]
complex_45deg = real_45deg + 1j * imag_45deg
phase_45deg = np.angle(complex_45deg)  # returns phase in radians (−π to π)

real_endfire = analytical_lines[:, 9]
imag_endfire = analytical_lines[:, 10]
abs_endfire = analytical_lines[:, 11]
complex_endfire = real_endfire + 1j * imag_endfire
phase_endfire = np.angle(complex_endfire)  # returns phase in radians (−π to π)

def plot_component_receiver_line(component, rec_x, rec_y, abs_data_list, phase_data_list, x_an, y_an, abs_an, phase_an, type_rec, index, purpose, labels):
    _, axes = plt.subplots(1, 2, figsize=(14, 6))
    type_string = 'y' if type_rec == 'Broadside' else 'x' if type_rec == 'Endfire' else 'Radial distance from the dipole in Oblique Configuration, 45°'

    # Style options
    colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728', '#6a0dad', '#8c564b']
    linestyles = ['--', ':', '--', ':', '--', ':']
    markers = ['o', 's', '^', 'v', 'X', 'D']
    marker_size = 8     # Bigger markers (default is ~6)
    line_width = 1.5    # Thinner lines (default is ~2)

    for i, (abs_data, phase_data, rec_xi, rec_yi, label) in enumerate(zip(abs_data_list, phase_data_list, rec_x, rec_y, labels)):
        rec_distance = rec_yi if type_rec == 'Broadside' else rec_xi if type_rec == 'Endfire' else np.sqrt(rec_xi**2 + rec_yi**2)
        style_idx = i % len(colors)
        # Absolute value plot
        axes[0].plot(rec_distance, abs_data,
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)
        # Phase plot
        axes[1].plot(rec_distance, phase_data,
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)

    # Analytical solution
    rec_distance_an = np.array(y_an) if type_rec == 'Broadside' else np.array(x_an) if type_rec == 'Endfire' else np.array(x_an)
    axes[0].plot(rec_distance_an, abs_an, linestyle='-', color='black', label='Analytical Solution', linewidth=2)
    axes[1].plot(rec_distance_an, phase_an, linestyle='-', color='black', label='Analytical Solution', linewidth=2)

    axes[0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0].set_ylabel(f'Abs({component})', fontsize=12)
    axes[0].set_title(f'Absolute Value of {component} Field Component', fontsize=18)
    axes[0].set_yscale('log')
    axes[0].grid(True)
    axes[0].legend(loc='upper right')

    axes[1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1].set_ylabel(f'Phase({component}) (degrees)', fontsize=12)
    axes[1].set_title(f'Phase of {component} Field Component', fontsize=18)
    axes[1].grid(True)

    plt.subplots_adjust(top=0.82)
    title = r'Half-Space Model Response - $\epsilon_r = 4$' \
            '\n{} Receiver Line Data of {} Component'.format(type_rec, component)
    plt.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    plot_name = f'{index}_{purpose}_{component}_receiver_line_{type_rec}.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

# Prepare data for each receiver type
rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list = [], [], [], []
rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list = [], [], [], []
rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list = [], [], [], []

for e_data_elfe in e_data_elfe_all:

    rec_x_elfe = e_data_elfe[:, 0]
    rec_y_elfe = e_data_elfe[:, 1]

    real_Ex_elfe = e_data_elfe[:, 4]
    imag_Ex_elfe = e_data_elfe[:, 5]
    abs_Ex_elfe = np.abs(real_Ex_elfe + 1j * imag_Ex_elfe)
    phase_Ex_elfe = np.angle(real_Ex_elfe + 1j * imag_Ex_elfe) # returns phase in radians (−π to π)

    # Broadside
    i_rx_bs = num_rec_ef
    rec_x_bs = rec_x_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_y_bs = rec_y_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    abs_Ex_bs = abs_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    phase_Ex_bs = phase_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_x_bs_list.append(rec_x_bs)
    rec_y_bs_list.append(rec_y_bs)
    abs_Ex_bs_list.append(abs_Ex_bs)
    phase_Ex_bs_list.append(phase_Ex_bs)

    # Endfire
    i_rx_ef = 0
    rec_x_ef = rec_x_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_y_ef = rec_y_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    abs_Ex_ef = abs_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    phase_Ex_ef = phase_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_x_ef_list.append(rec_x_ef)
    rec_y_ef_list.append(rec_y_ef)
    abs_Ex_ef_list.append(abs_Ex_ef)
    phase_Ex_ef_list.append(phase_Ex_ef)

    # Oblique
    i_rx_ob = num_rec_bs + num_rec_ef
    rec_x_ob = rec_x_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_y_ob = rec_y_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    abs_Ex_ob = abs_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    phase_Ex_ob = phase_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_x_ob_list.append(rec_x_ob)
    rec_y_ob_list.append(rec_y_ob)
    abs_Ex_ob_list.append(abs_Ex_ob)
    phase_Ex_ob_list.append(phase_Ex_ob)

# Analytical data
# -------------------------------------
# Got the broadside and endire mixed up, so now we have to fix it by re-assigning to the right
# variables.
# -------------------------------------
x_an_ef = r
y_an_ef = np.zeros_like(r)
abs_Ex_an_ef = abs_broadside
phase_Ex_an_ef = phase_broadside

x_an_bs = np.zeros_like(r)
y_an_bs = r
abs_Ex_an_bs = abs_endfire
phase_Ex_an_bs = phase_endfire

x_an_ob = r
y_an_ob = r
abs_Ex_an_ob = abs_45deg
phase_Ex_an_ob = phase_45deg

# Plot all on the same axes for each receiver type
plot_component_receiver_line('Ex', rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, 
                             x_an_bs, y_an_bs, abs_Ex_an_bs, phase_Ex_an_bs, 'Broadside', 8, '4_9', labels)
plot_component_receiver_line('Ex', rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, 
                             x_an_ef, y_an_ef, abs_Ex_an_ef, phase_Ex_an_ef, 'Endfire', 9, '4_9', labels)
plot_component_receiver_line('Ex', rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, 
                             x_an_ob, y_an_ob, abs_Ex_an_ob, phase_Ex_an_ob, 'Oblique, 45°', 10, '4_9', labels)

In [5]:
electric_file = os.path.join(base_folder, "out_4_9_ref_Berenger", "electric_fields_receiver_line.txt")
e_data_elfe_1 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_9_ref_Berenger2", "electric_fields_receiver_line.txt")
e_data_elfe_2 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_9_ref_Feng", "electric_fields_receiver_line.txt")
e_data_elfe_3 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_9_bc_atomic", "electric_fields_receiver_line.txt")
e_data_elfe_4 = np.loadtxt(electric_file)

e_data_elfe_all = [e_data_elfe_1, e_data_elfe_2, e_data_elfe_3, e_data_elfe_4]

labels = ['Berenger - kmax+eps_max',
          'Berenger - kmax+eps_var',
          'Feng - k+eps var',
          'Feng - k+eps var, || DBC'
          ]

num_rec_bs=15
num_rec_ef=15
num_rec_ob=15

# Analytical Data
analytical_file = os.path.join(analytical_folder, "Exx_single_freq_4_9_100MHz.csv")
analytical_lines = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

r = analytical_lines[:, 0]

real_broadside = analytical_lines[:, 1]
imag_broadside = analytical_lines[:, 2]
abs_broadside = analytical_lines[:, 3]
complex_broadside = real_broadside + 1j * imag_broadside
phase_broadside = np.angle(complex_broadside)  # returns phase in radians (−π to π)

real_45deg = analytical_lines[:, 5]
imag_45deg = analytical_lines[:, 6]
abs_45deg = analytical_lines[:, 7]
complex_45deg = real_45deg + 1j * imag_45deg
phase_45deg = np.angle(complex_45deg)  # returns phase in radians (−π to π)

real_endfire = analytical_lines[:, 9]
imag_endfire = analytical_lines[:, 10]
abs_endfire = analytical_lines[:, 11]
complex_endfire = real_endfire + 1j * imag_endfire
phase_endfire = np.angle(complex_endfire)  # returns phase in radians (−π to π)

def plot_component_receiver_line(component, rec_x, rec_y, abs_data_list, phase_data_list, x_an, y_an, abs_an, phase_an, type_rec, index, purpose, labels):
    _, axes = plt.subplots(1, 2, figsize=(14, 6))
    type_string = 'y' if type_rec == 'Broadside' else 'x' if type_rec == 'Endfire' else 'Radial distance from the dipole in Oblique Configuration, 45°'

    # Style options
    colors = ['#1f77b4', '#2ca02c', '#d62728', '#6a0dad']
    linestyles = [':', ':', ':', ':', ':']
    markers = ['o', 's', 'v', 'X']
    marker_size = 8     # Bigger markers (default is ~6)
    line_width = 1.5    # Thinner lines (default is ~2)

    for i, (abs_data, phase_data, rec_xi, rec_yi, label) in enumerate(zip(abs_data_list, phase_data_list, rec_x, rec_y, labels)):
        rec_distance = rec_yi if type_rec == 'Broadside' else rec_xi if type_rec == 'Endfire' else np.sqrt(rec_xi**2 + rec_yi**2)
        style_idx = i % len(colors)
        # Absolute value plot
        axes[0].plot(rec_distance, abs_data,
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)
        # Phase plot
        axes[1].plot(rec_distance, phase_data,
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)


    # Analytical solution
    rec_distance_an = np.array(y_an) if type_rec == 'Broadside' else np.array(x_an) if type_rec == 'Endfire' else np.array(x_an)
    axes[0].plot(rec_distance_an, abs_an, linestyle='-', color='black', label='Analytical Solution')
    axes[1].plot(rec_distance_an, phase_an, linestyle='-', color='black', label='Analytical Solution')

    axes[0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0].set_ylabel(f'Abs({component})', fontsize=12)
    axes[0].set_title(f'Absolute Value of {component} Field Component', fontsize=18)
    axes[0].set_yscale('log')
    axes[0].grid(True)
    axes[0].legend(loc='upper right')

    axes[1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1].set_ylabel(f'Phase({component}) (degrees)', fontsize=12)
    axes[1].set_title(f'Phase of {component} Field Component', fontsize=18)
    axes[1].grid(True)
    # axes[1].legend(loc='upper right')

    plt.subplots_adjust(top=0.82)  # Give more space for the suptitle
    title = r'Two-Layered Model Response - 2 m Layer of $\epsilon_r = 4$ with Half-space of $\epsilon_r = 9$' \
            '\n{} Receiver Line Data of {} Component'.format(type_rec, component)
    plt.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    plot_name = f'{index}_{purpose}_{component}_receiver_line_{type_rec}.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

# Prepare data for each receiver type
rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list = [], [], [], []
rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list = [], [], [], []
rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list = [], [], [], []

for e_data_elfe in e_data_elfe_all:

    rec_x_elfe = e_data_elfe[:, 0]
    rec_y_elfe = e_data_elfe[:, 1]

    real_Ex_elfe = e_data_elfe[:, 4]
    imag_Ex_elfe = e_data_elfe[:, 5]
    abs_Ex_elfe = np.abs(real_Ex_elfe + 1j * imag_Ex_elfe)
    phase_Ex_elfe = np.angle(real_Ex_elfe + 1j * imag_Ex_elfe) # returns phase in radians (−π to π)

    # Broadside
    i_rx_bs = num_rec_ef
    rec_x_bs = rec_x_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_y_bs = rec_y_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    abs_Ex_bs = abs_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    phase_Ex_bs = phase_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_x_bs_list.append(rec_x_bs[:-1])
    rec_y_bs_list.append(rec_y_bs[:-1])
    abs_Ex_bs_list.append(abs_Ex_bs[:-1])
    phase_Ex_bs_list.append(phase_Ex_bs[:-1])

    # Endfire
    i_rx_ef = 0
    rec_x_ef = rec_x_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_y_ef = rec_y_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    abs_Ex_ef = abs_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    phase_Ex_ef = phase_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_x_ef_list.append(rec_x_ef[:-1])
    rec_y_ef_list.append(rec_y_ef[:-1])
    abs_Ex_ef_list.append(abs_Ex_ef[:-1])
    phase_Ex_ef_list.append(phase_Ex_ef[:-1])

    # Oblique
    i_rx_ob = num_rec_bs + num_rec_ef
    rec_x_ob = rec_x_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_y_ob = rec_y_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    abs_Ex_ob = abs_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    phase_Ex_ob = phase_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_x_ob_list.append(rec_x_ob[:-1])
    rec_y_ob_list.append(rec_y_ob[:-1])
    abs_Ex_ob_list.append(abs_Ex_ob[:-1])
    phase_Ex_ob_list.append(phase_Ex_ob[:-1])

# Analytical data
# -------------------------------------
# Got the broadside and endire mixed up, so now we have to fix it by re-assigning to the right
# variables.
# -------------------------------------
x_an_ef = r
y_an_ef = np.zeros_like(r)
abs_Ex_an_ef = abs_broadside
phase_Ex_an_ef = phase_broadside

x_an_bs = np.zeros_like(r)
y_an_bs = r
abs_Ex_an_bs = abs_endfire
phase_Ex_an_bs = phase_endfire

x_an_ob = r
y_an_ob = r
abs_Ex_an_ob = abs_45deg
phase_Ex_an_ob = phase_45deg

# Plot all on the same axes for each receiver type
plot_component_receiver_line('Ex', rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, 
                             x_an_bs, y_an_bs, abs_Ex_an_bs, phase_Ex_an_bs, 'Broadside', 11, '4_9', labels)
plot_component_receiver_line('Ex', rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, 
                             x_an_ef, y_an_ef, abs_Ex_an_ef, phase_Ex_an_ef, 'Endfire', 12, '4_9', labels)
plot_component_receiver_line('Ex', rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, 
                             x_an_ob, y_an_ob, abs_Ex_an_ob, phase_Ex_an_ob, 'Oblique, 45°', 13, '4_9', labels)